# P04 · Notebook 25 · A second view for the single-microphone system (DCASE 2020: fans and pumps)

**Where we stand** (notebook 23, the frozen MIC system on DCASE 2020, AUC; bold is a weak point):

| Set | Pump | Fan | Valve | Slide rail |
|---|---|---|---|---|
| Development | 0.9726 | **0.9074** | 0.9977 | 0.9930 |
| Evaluation | **0.9440** | **0.9593** | 0.9744 | 0.9991 |

**What it does:**
1. **Checks first (minutes).** Notebook 23's files in Drive are the recorded ones, and the frozen system's numbers come back from its saved embeddings. No BEATs or ECHO model is loaded.
2. **Trains a second view from scratch, three seeds.** TASTgram-MFN with Noisy-ArcMix (ICASSP 2024), the published recipe, on DCASE 2020's own training clips only.
3. **Fuses it with the frozen system in five fixed forms** and selects one on development pumps and fans. It is kept only if the development score rises by 0.01 and valves and slide rails lose no more than 0.01.
4. **Only if it is kept:** the evaluation set is scored once by the kept system. The scores are saved before any label is read, and the official evaluator checks the numbers.

**Rules:** `results/mimii_beats/RULES_second_view_written_before_running.md`, written before this notebook was built, and amended once (Amendment 1) while it was tested on synthetic clips, before any run on real data.

**It needs** in Google Drive: `P04_ExternalJudgement` (notebook 23). It writes to a new folder, `P04_SecondView` (1 GB is enough).

**Runtime:**
- Choose **G4** (RTX PRO 6000): Runtime › Change runtime type. Then Runtime › Run all.
- About 7.8 GB of downloads, then three trainings of 300 epochs.
- **Time (my estimate, not measured):** about 1–3 hours on G4; about 6–24 hours on a T4. Cell 7 measures this GPU in the first minute of training and prints its own estimate.
- Everything is saved to Drive. If Colab disconnects, run all again: finished trainings are skipped, and an unfinished one continues from its last save (every 10 epochs).

**At the end** `p04_n25_results.zip` downloads. Send it back here.

In [ ]:
# 1 · Setup
import os, sys, json, time, hashlib, struct, zipfile, subprocess, shutil, urllib.request, re, math, gc, csv
from pathlib import Path
from collections import deque
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Parameter
from sklearn.metrics import roc_auc_score
try:
    import torchaudio
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchaudio"], check=True)
    import torchaudio

TEST = os.environ.get("P04_TEST") == "1"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    MY = Path("/content/drive/MyDrive")
    OUT, WORK, N23 = MY / "P04_SecondView", Path("/content/work"), MY / "P04_ExternalJudgement"
else:
    OUT, WORK = Path(os.environ.get("P04_OUT", "P04_SecondView")), Path(os.environ.get("P04_WORK", "work"))
    N23 = Path(os.environ.get("P04_N23_DIR", "P04_ExternalJudgement"))
if not N23.exists():
    raise SystemExit(f"missing {N23}: notebook 23's folder (P04_ExternalJudgement) must be in Google Drive › MyDrive")
for p in (OUT, WORK):
    p.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP = DEVICE == "cuda"
GPU = torch.cuda.get_device_name(0) if DEVICE == "cuda" else ""
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE == "cuda" else 0.0
if DEVICE != "cuda" and not TEST:
    raise SystemExit("No GPU. Runtime › Change runtime type › G4 (or another GPU), then Runtime › Run all.")
if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True                             # fixed input sizes: cuDNN picks its fastest algorithms
T_START = time.time()
DRIVE_FREE_GB = shutil.disk_usage(OUT).free / 1e9
print("device:", DEVICE, GPU, f"({GPU_GB:.0f} GB)" if GPU else "", "· torch", torch.__version__, "· torchaudio", torchaudio.__version__,
      f"· free local disk {shutil.disk_usage(WORK).free / 1e9:.0f} GB · free Drive {DRIVE_FREE_GB:.1f} GB")


def md5(path, chunk=8 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def sha256(path, chunk=8 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def download(url, dest):
    dest = Path(dest); part = dest.with_suffix(dest.suffix + ".part")
    try:
        if url.startswith("file://"):
            shutil.copy(url[7:], part)
        elif shutil.which("wget"):
            subprocess.run(["wget", "-q", "--show-progress", "-c", "-O", str(part), url], check=True)
        else:
            urllib.request.urlretrieve(url, part)
        part.rename(dest)
    finally:
        if part.exists() and not dest.exists():
            part.unlink()


download_one = download                              # notebook 11's single-connection download


def download(url, dest):
    # several connections per file with aria2c (notebook 19 saw about 2.4 MB/s per connection from Zenodo); falls back to one connection
    dest = Path(dest); part = dest.with_suffix(dest.suffix + ".part")
    if not url.startswith("file://"):
        if shutil.which("aria2c") is None and IN_COLAB:
            if subprocess.run(["apt-get", "-qq", "install", "-y", "aria2"], capture_output=True).returncode != 0:
                subprocess.run(["apt-get", "-qq", "update"], capture_output=True)
                subprocess.run(["apt-get", "-qq", "install", "-y", "aria2"], capture_output=True)
        if shutil.which("aria2c"):
            r = subprocess.run(["aria2c", "-x", "4", "-s", "4", "-k", "8M", "--file-allocation=none", "--max-tries=30", "--retry-wait=10",
                                "--allow-overwrite=true", "--auto-file-renaming=false", "--console-log-level=warn", "--summary-interval=0",
                                "--download-result=hide", "-d", str(dest.parent), "-o", part.name, url])
            if r.returncode == 0 and part.exists():
                part.rename(dest)
                return
            print(f"  aria2c failed for {dest.name} (code {r.returncode}); one connection instead", flush=True)
            for f in (part, Path(str(part) + ".aria2")):
                f.unlink(missing_ok=True)
    download_one(url, dest)

In [ ]:
# 2 · Every number below was fixed in the rules before this notebook ran
RULES = "results/mimii_beats/RULES_second_view_written_before_running.md"
# DCASE 2020 Task 2 on Zenodo: the additional training data, then development, evaluation and ground truth (notebook 23's lines)
ZX = "https://zenodo.org/records/3727685/files/"                    # DCASE 2020 Task 2, additional training dataset
XMD5 = {"pump": "4e33ae9c0db5cc88437675f5e317caee", "fan": "567798854130c8019df8b664c095ed1e",
        "valve": "057cdd560caffb8305449541afd28c6d", "slider": "f84f186f4295fc23754abcdd1b0580a7"}
XZIPS = {t_: (ZX + f"eval_data_train_{t_}.zip?download=1", f"eval_data_train_{t_}.zip", h) for t_, h in XMD5.items()}
ZD, ZE, ZG = ("https://zenodo.org/records/3678171/files/", "https://zenodo.org/records/3841772/files/",
              "https://zenodo.org/records/3951620/files/")         # DCASE 2020 Task 2: development, evaluation, ground truth
DMD5 = {"fan": "649bdfc06263ae7a838963f43b6641e6", "pump": "90e7091ef722b7238a7f1009365779cd",
        "slider": "da24a757719f0d94d5aa2d646bbfdc86", "valve": "34d2672f55bb041589ef79c10dc89934"}
EMD5 = {"fan": "1eb9356a768cadfd0f2e59a5c57e578b", "pump": "23a8f8f924be218c69df67fe07360348",
        "slider": "0193c769073840332ce3aad84b1ccaa2", "valve": "f69c551d2088691050d20cccca9d631c"}
DC_DEV = {t_: (ZD + f"dev_data_{t_}.zip?download=1", f"dev_data_{t_}.zip", h) for t_, h in DMD5.items()}
DC_EVAL = {t_: (ZE + f"eval_data_test_{t_}.zip?download=1", f"eval_data_test_{t_}.zip", h) for t_, h in EMD5.items()}
DC_GT = (ZG + "eval_data_list.csv?download=1", "eval_data_list.csv", "f16aaa185df9906e588864aaf7cb58ea")
TYPES, MAIN, AUX = ["pump", "fan", "valve", "slider"], ["pump", "fan"], ["valve", "slider"]
SR, LDN_FLOOR, BAR = 16000, 1e-6, 0.01
# the second view: the settings of github.com/soonhyeon/Noisy-ArcMix at REF_COMMIT (config.yaml and the code)
REF_COMMIT = "674c759bfaa7723619a4b1d8d00eb9d155c50ba6"
SV = dict(n_fft=2048, hop=512, n_mels=128, win_len=1024, c_dim=128, arc_s=30.0, arc_m=0.7, alpha=0.5, lr=1e-4, eta_min_ratio=0.1,
          epochs=300, batch=64, split_ratio=0.95, seeds=[0, 1, 2], sine_floor=1e-9, save_every=10, embed_batch=64,
          gpu_headroom_gb=5.0)
REF_FRAMES, REF_KERNEL, REF_CLASSES, REF_CLIPS, REF_TRAIN = 313, (8, 20), 28, 23283, 22118
FORMS = ["A", "G", "C", "Z", "ZP"]                                    # mean of scores, geometric mean, concatenation, and the two calibrated
                                                                      # sums of Amendment 1 (the view by LDN1, the view by the paper's score)
FROZEN_CAP, CAL_MIN = 600, 30                                         # form Z (Amendment 1): notebook 23 trained on the first 600 additional clips of a
                                                                      # machine; a spread needs at least 30 clips per type
FROZEN = [("BEATs", "B", [0, 1, 2]), ("BEATs-iter3", "B3", [0, 1, 2])]   # notebook 23's final system: encoder, file code, seeds
CURRENT, KEEP_AT = (0.94, 0.9953), (0.95, 0.9853)                                 # notebook 23 (development, confirmation); the keep rule's two bars
G0C_TOL, G0C_REL, EVAL_TOL, DRIVE_MIN_GB, DISK_MIN_GB = 0.0005, 1e-6, 1e-9, 1.0, 15.0
# notebook 23's results, from its results zip (MD5 8301df3f5c591db368a94265fd352cef): the files gate G0(a) checks in Drive, and the
# numbers gates G0(b) and G0(c) compare with (Amendment 1: Drive's copy of the main JSON need not be byte-identical)
N23_JSON_MD5, N23_RESULTS_MD5 = "2e5b735796eb4029d5af8cad1d47d673", "afc5c84df7271303189c5bd4bbff459e"
SCORE_MD5 = {"anomaly_score_fan_id_01.csv": "4479511bf06757a6fefd1e04c6ad8b9d",
             "anomaly_score_fan_id_03.csv": "42576aeb8791334689d44804f5d87729",
             "anomaly_score_fan_id_05.csv": "1e8dc59ca00f27f39c1cbfbf902065a3",
             "anomaly_score_pump_id_01.csv": "3130aa04f5f9778fad3eafcac6abc0b6",
             "anomaly_score_pump_id_03.csv": "787e4eac75186f020b836c58176e3a30",
             "anomaly_score_pump_id_05.csv": "ef7e6993f3e133bff5eb5026bd5aa26f",
             "anomaly_score_slider_id_01.csv": "75e8c848ab2ca8793614aceef3fba1f8",
             "anomaly_score_slider_id_03.csv": "54be22b8c7e90a35257c9f8333a67213",
             "anomaly_score_slider_id_05.csv": "dc8d0536f6078e45e74dea882b638871",
             "anomaly_score_valve_id_01.csv": "1fd7c7a2bd54669c3b9128131f04ea31",
             "anomaly_score_valve_id_03.csv": "3a2800dc6f282f23c03e841bfb8b76c0",
             "anomaly_score_valve_id_05.csv": "93b3c6a7fef7c75f66452924af56c9ee"}
N23_DEV = {"pump": [0.972641, 0.935595], "fan": [0.907373, 0.833838], "valve": [0.997729, 0.989693], "slider": [0.992963, 0.971196]}      # notebook 23, development: AUC, pAUC
N23_EVAL = {"pump": [0.944015, 0.855696], "fan": [0.959277, 0.90921], "valve": [0.974425, 0.937939], "slider": [0.999101, 0.99655]}     # notebook 23, evaluation (scored once): AUC, pAUC
# clips per type, set and machine; and per development machine: AUC, pAUC, test clips, anomalies (full precision)
N23_COUNTS = {"pump": {"dev|train": {"id_00": 906, "id_02": 905, "id_04": 602, "id_06": 936}, "dev|test": {"id_00": 243, "id_02": 211, "id_04": 200, "id_06": 202}, "eval|train": {"id_01": 903, "id_03": 606, "id_05": 908}, "eval|test": {"id_01": 216, "id_03": 213, "id_05": 348}},
              "fan": {"dev|train": {"id_00": 911, "id_02": 916, "id_04": 933, "id_06": 915}, "dev|test": {"id_00": 507, "id_02": 459, "id_04": 448, "id_06": 461}, "eval|train": {"id_01": 934, "id_03": 912, "id_05": 1000}, "eval|test": {"id_01": 426, "id_03": 458, "id_05": 458}},
              "valve": {"dev|train": {"id_00": 891, "id_02": 608, "id_04": 900, "id_06": 892}, "dev|test": {"id_00": 219, "id_02": 220, "id_04": 220, "id_06": 220}, "eval|train": {"id_01": 769, "id_03": 863, "id_05": 899}, "eval|test": {"id_01": 220, "id_03": 220, "id_05": 500}},
              "slider": {"dev|train": {"id_00": 968, "id_02": 968, "id_04": 434, "id_06": 434}, "dev|test": {"id_00": 456, "id_02": 367, "id_04": 278, "id_06": 189}, "eval|train": {"id_01": 968, "id_03": 968, "id_05": 434}, "eval|test": {"id_01": 278, "id_03": 278, "id_05": 278}}}
N23_DEV_PM = {"pump": {'id_00': [0.9453846153846155, 0.8623481781376519, 243, 143], 'id_02': [0.9605405405405405, 0.9217638691322902, 211, 111], 'id_04': [0.9916, 0.9768421052631577, 200, 100], 'id_06': [0.9930392156862746, 0.9814241486068112, 202, 102]},
              "fan": {'id_00': [0.7329238329238329, 0.5870942713047976, 507, 407], 'id_02': [0.9807242339832869, 0.9371059961882422, 459, 359], 'id_04': [0.9160919540229885, 0.8124621899576527, 448, 348], 'id_06': [0.9997506925207756, 0.9986878553725034, 461, 361]},
              "valve": {'id_00': [1.0, 1.0, 219, 119], 'id_02': [0.9999166666666667, 0.999561403508772, 220, 120], 'id_04': [0.9985, 0.9956140350877194, 220, 120], 'id_06': [0.9924999999999999, 0.9635964912280702, 220, 120]},
              "slider": {'id_00': [1.0, 1.0, 456, 356], 'id_02': [0.9873033707865168, 0.9660950128129312, 367, 267], 'id_04': [0.9979213483146067, 0.9890597279716145, 278, 178], 'id_06': [0.986629213483146, 0.9296274393849793, 189, 89]}}
DCASE_BASELINE_DEV = {"fan": 0.6583, "pump": 0.7289, "slider": 0.8476, "valve": 0.6628}   # the task page, mean of 10 trials (gate G-VIEW here)
TEAMS = {"Giri_Amazon_task2_2 (rank 1)": {"fan": (94.54, 84.30), "pump": (93.65, 81.73), "slider": (97.63, 89.73), "valve": (96.13, 90.89)},
         "Daniluk_SRPOL_task2_4 (rank 3)": {"fan": (99.13, 96.40), "pump": (95.07, 90.23), "slider": (98.18, 91.98), "valve": (90.97, 77.41)},
         "Primus_CP-JKU_task2_2 (rank 5)": {"fan": (96.84, 95.24), "pump": (97.76, 92.24), "slider": (97.29, 88.74), "valve": (90.15, 86.65)},
         "Vinayavekhin_IBM_task2_2 (rank 7)": {"fan": (98.84, 94.89), "pump": (94.37, 88.27), "slider": (95.68, 83.09), "valve": (97.82, 94.93)},
         "Zhou_PSH_task2_3 (rank 12)": {"fan": (99.79, 98.92), "pump": (95.79, 92.60), "slider": (99.84, 99.17), "valve": (91.83, 84.74)}}
BEST5 = {t_: max(v[t_][0] for v in TEAMS.values()) for t_ in TYPES}   # the best of the program's five teams per type (evaluation AUC %)
# published numbers on the development set, AUC and pAUC % (41 classes; yardsticks, not decision bars)
PUBLISHED_DEV = {'Noisy-ArcMix, TASTgram-MFN (arXiv 2310.06364)': {'fan': (98.32, 95.34), 'pump': (95.44, 85.99), 'slider': (99.53, 97.5), 'valve': (99.95, 99.74)},
                 'STgram-MFN with ArcFace (same table)': {'fan': (94.04, 88.97), 'pump': (91.94, 81.75), 'slider': (99.55, 97.61), 'valve': (99.64, 98.44)},
                 'FGASpecNet (Mathematics 2026, 14(3), 530)': {'fan': (98.05, 94.73), 'pump': (96.03, 88.24), 'slider': (99.49, 97.32), 'valve': (99.93, 99.61)}}
PUBLISHED_MAUC = {'fan': 92.67, 'pump': 91.17, 'slider': 97.96, 'valve': 99.89}   # Noisy-ArcMix: the worst machine of each type
# AnoPatch, arXiv 2406.11364, Table 1: the mean of AUC and pAUC, %
ANOPATCH = {'development': {'fan': 86.46, 'pump': 93.1, 'slider': 99.2, 'valve': 97.53},
            'evaluation': {'fan': 95.56, 'pump': 94.34, 'slider': 99.77, 'valve': 96.26}}
# the official evaluator, pinned to one commit and checked by its SHA-256 (as in notebook 24)
EVALUATOR = ("https://raw.githubusercontent.com/y-kawagu/dcase2020_task2_evaluator/3e5464fe3091cc647015a84475eb89c3232459b6/evaluator.py",
             "evaluator.py", "1a514d3809b6aabab60db7fa4067d0e59a97c98c728918427a546763974e106a")
FORCE = None
if TEST:
    XZIPS = {t_: tuple(v) for t_, v in json.loads(os.environ["P04_TEST_DATA"])["xc"].items()}
    _dc = json.loads(os.environ["P04_TEST_DCASE"])
    DC_DEV, DC_EVAL, DC_GT = {t_: tuple(v) for t_, v in _dc["dev"].items()}, {t_: tuple(v) for t_, v in _dc["eval"].items()}, tuple(_dc["gt"])
    _n = json.loads(os.environ["P04_TEST_N23"])
    N23_JSON_MD5, N23_RESULTS_MD5, SCORE_MD5, EVALUATOR = _n["json_md5"], _n["results_md5"], _n["score_md5"], tuple(_n["evaluator"])
    _j = json.loads((N23 / "external_judgement.json").read_text())["dcase"]      # the test's stand-in for notebook 23 carries its own numbers
    N23_COUNTS = _j["counts"]
    N23_DEV_PM = {t_: {q: [v["AUC"], v["pAUC"], v["n_test"], v["n_anomalies"]] for q, v in _j["development"]["final"][t_]["per_machine"].items()} for t_ in TYPES}
    N23_DEV = {t_: [_j["development"]["final"][t_][k] for k in ("AUC", "pAUC")] for t_ in TYPES}
    N23_EVAL = {t_: [_j["evaluation"]["per_type"][t_][k] for k in ("AUC", "pAUC")] for t_ in TYPES}
    _c = json.loads(os.environ.get("P04_TEST_CFG", "{}"))
    SV.update(_c.get("sv", {})); FROZEN_CAP, CAL_MIN = _c.get("frozen_cap", FROZEN_CAP), _c.get("cal_min", CAL_MIN)
    FORCE = os.environ.get("P04_TEST_FORCE")                          # tests only: walk the kept / not-kept branch
print("rules:", RULES, "·", json.dumps(SV))
if not TEST and DRIVE_FREE_GB < DRIVE_MIN_GB:
    raise SystemExit(f"stop: Drive has {DRIVE_FREE_GB:.1f} GB free and this run needs {DRIVE_MIN_GB} GB. Free some space in Google Drive, then run again.")
if not TEST and shutil.disk_usage(WORK).free / 1e9 < DISK_MIN_GB:
    raise SystemExit(f"stop: the local disk has {shutil.disk_usage(WORK).free / 1e9:.0f} GB free and this run needs {DISK_MIN_GB:.0f} GB")

In [ ]:
# 3 · DCASE 2020's own files (MD5 against the Zenodo records) and every clip of each type in one int16 store on the local disk.
# The loader is notebook 23's, unchanged, so the clips come in the order of the embeddings notebook 23 saved (gate G0(c) proves it).
CHECKED = {}                                        # files whose MD5 was already checked in this session


def get_file(entry):
    url, name, want = entry
    dest = WORK / name
    if CHECKED.get(name) == want and dest.exists():
        return dest
    if dest.exists() and md5(dest) == want:
        CHECKED[name] = want
        return dest
    t = time.time(); download(url, dest)
    got = md5(dest)
    print(f"{name}: downloaded in {time.time() - t:.0f} s · MD5 matches Zenodo: {got == want}", flush=True)
    if got != want:
        dest.unlink()
        raise SystemExit(f"stop: {name} is not the published file")
    CHECKED[name] = want
    return dest


def read_all(data):
    # WAV bytes (plain or WAVE_FORMAT_EXTENSIBLE, 16-bit PCM) -> (samples, channels) float32 in [-1, 1], sample rate
    assert data[:4] == b"RIFF" and data[8:12] == b"WAVE", "not a WAV file"
    pos, fmt, raw = 12, None, None
    while pos + 8 <= len(data):
        cid, size = data[pos:pos + 4], struct.unpack("<I", data[pos + 4:pos + 8])[0]
        body = data[pos + 8:pos + 8 + size]
        if cid == b"fmt ":
            tag, ch, sr, _, _, bits = struct.unpack("<HHIIHH", body[:16])
            if tag == 0xFFFE:
                tag = struct.unpack("<H", body[24:26])[0]
            fmt = (tag, ch, sr, bits)
        elif cid == b"data":
            raw = body
        pos += 8 + size + (size & 1)
    tag, ch, sr, bits = fmt
    assert tag == 1 and bits == 16, f"expected 16-bit PCM, got format {tag}, {bits}-bit"
    nf = len(raw) // (2 * ch)
    return np.frombuffer(raw[:nf * 2 * ch], dtype="<i2").reshape(-1, ch).astype(np.float32) / 32768.0, sr


def stream(fn, items, depth=8, workers=2):
    # fn(item) runs in background threads, results come back in order: unzipping overlaps the main thread's work
    with ThreadPoolExecutor(workers) as ex:
        q, it = deque(), iter(items)
        for item in it:
            q.append(ex.submit(fn, item))
            if len(q) >= depth:
                break
        while q:
            res = q.popleft().result()
            for item in it:
                q.append(ex.submit(fn, item))
                break
            yield res


def to_int16(v):
    # only for microphone-0 samples read from the 16-bit WAV files, so this conversion is exact
    return np.clip(np.round(v * 32768.0), -32768, 32767).astype(np.int16)


def f32(a):
    # a slice of a stream → float32 signal: int16 streams hold samples × 32768, float16 streams hold the signal itself
    a = np.asarray(a)
    return a.astype(np.float32) / 32768.0 if a.dtype == np.int16 else a.astype(np.float32)


STORE = WORK / "stores25"; STORE.mkdir(exist_ok=True)
DST, DMETA = {}, {}


def dcase_load(t):
    # every DCASE 2020 clip of type t in one int16 stream (each clip is microphone 0): development train and test, the additional
    # training clips (all of them, not only XC's first 600) and the evaluation test clips
    meta_f, arr_f = STORE / f"dc_{t}.json", STORE / f"dc_{t}.npy"
    if meta_f.exists() and arr_f.exists():
        DMETA[t] = json.loads(meta_f.read_text()); DST[t] = np.load(arr_f, mmap_mode="r")
        return
    zfs = {"dev": zipfile.ZipFile(get_file(DC_DEV[t])), "evaltrain": zipfile.ZipFile(get_file(XZIPS[t])), "evaltest": zipfile.ZipFile(get_file(DC_EVAL[t]))}
    rows = []
    for kind, zf in zfs.items():
        for nm in sorted(x for x in zf.namelist() if x.lower().endswith(".wav")):
            parts = nm.replace("\\", "/").split("/"); base, folder = parts[-1], parts[-2]
            part = "train" if folder == "train" else "test"
            label = (1 if base.startswith("anomaly") else 0) if (kind == "dev" and part == "test") else (0 if part == "train" else -1)
            rows.append((kind, nm, "dev" if kind == "dev" else "eval", part, re.search(r"(id_\d+)", base).group(1), label, base))
    ids = {(s_, p_): sorted(set(r[4] for r in rows if r[2] == s_ and r[3] == p_)) for s_ in ("dev", "eval") for p_ in ("train", "test")}
    if not TEST:
        assert ids[("dev", "train")] == ids[("dev", "test")] == ["id_00", "id_02", "id_04", "id_06"], ids
        assert ids[("eval", "train")] == ids[("eval", "test")] == ["id_01", "id_03", "id_05"], ids
    assert not any(r[2] == "eval" and r[3] == "train" and not r[6].startswith("normal") for r in rows), "the additional training clips are normal"
    ns = read_all(zfs[rows[0][0]].read(rows[0][1]))[0].shape[0]
    if not TEST:
        assert ns == 10 * SR, ns
    tmp = arr_f.with_name(arr_f.stem + ".part.npy")
    mm = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.int16, shape=(len(rows), ns))
    for i, (x, sr) in enumerate(stream(lambda i: read_all(zfs[rows[i][0]].read(rows[i][1])), range(len(rows)))):
        assert sr == SR and x.shape[0] == ns, (rows[i][1], sr, x.shape)
        mm[i] = to_int16(x[:, 0])
    mm.flush(); del mm
    os.replace(tmp, arr_f)
    for zf in zfs.values():
        zf.close()
    DMETA[t] = {"path": [r[1] for r in rows], "name": [r[6] for r in rows], "set": [r[2] for r in rows], "part": [r[3] for r in rows],
                "mid": [r[4] for r in rows], "label": [r[5] for r in rows], "samples": int(ns),
                "zips": {"dev": DC_DEV[t][1:], "additional": XZIPS[t][1:], "evaluation": DC_EVAL[t][1:]}}
    meta_f.write_text(json.dumps(DMETA[t]))
    DST[t] = np.load(arr_f, mmap_mode="r")
    if IN_COLAB:
        for z in (DC_DEV[t], XZIPS[t], DC_EVAL[t]):
            (WORK / z[1]).unlink(missing_ok=True)


def dc_rows(t, set_, part, q=None):
    m = DMETA[t]
    return np.array([i for i in range(len(m["name"])) if m["set"][i] == set_ and m["part"][i] == part and (q is None or m["mid"][i] == q)], dtype=np.int64)


t_data = time.time()
DCASE = {"counts": {}}
for t in TYPES:
    dcase_load(t)
    DCASE["counts"][t] = {f"{s_}|{p_}": {q: int(len(dc_rows(t, s_, p_, q))) for q in sorted(set(np.array(DMETA[t]["mid"])[dc_rows(t, s_, p_)].tolist()))}
                          for s_ in ("dev", "eval") for p_ in ("train", "test")}
    print(f"DCASE 2020 · {t}: {len(DMETA[t]['name'])} clips {DCASE['counts'][t]}", flush=True)
NS = int(DMETA[TYPES[0]]["samples"])
assert all(int(DMETA[t]["samples"]) == NS for t in TYPES), "the types' clips differ in length"
PREP_S = round(time.time() - t_data, 1)
print(f"data ready · {NS} samples per clip · {PREP_S / 60:.1f} min")

In [ ]:
# 4 · Gate G0: notebook 23's files, the clip counts, and the frozen system recomputed from notebook 23's saved embeddings.
# No BEATs or ECHO model is loaded: the frozen system's embeddings of every DCASE clip are read from Drive › P04_ExternalJudgement.
def unit(E):
    return E / np.linalg.norm(E, axis=-1, keepdims=True)


def b1(R, X):
    return 1.0 - (unit(X) @ unit(R).T).max(1)


def ldn1(R, X):
    # Wilkinghoff et al. (TASLP 2026), ratio form with K = 1: min over references y of d(x, y) / d(y, y₁)
    U = unit(R); Srr = U @ U.T; np.fill_diagonal(Srr, -np.inf)
    d1 = np.maximum(1.0 - Srr.max(1), LDN_FLOOR)
    return ((1.0 - unit(X) @ U.T) / d1[None, :]).min(1)


def dc_metrics(y, s):
    # as the official evaluator: sklearn's AUC, and its pAUC with max_fpr = 0.1 (standardised)
    return {"AUC": float(roc_auc_score(y, s)), "pAUC": float(roc_auc_score(y, s, max_fpr=0.1)), "n_test": int(len(y)), "n_anomalies": int(np.sum(y))}


# G0(a) · notebook 23's evaluation score files and results in Drive are the recorded ones (by MD5). Its main JSON is compared too,
# without stopping: running notebook 23's last cell again would rewrite it with a new date (Amendment 1)
_ev = N23 / "dcase2020_evaluation"
_bad = [f for f, h in {**SCORE_MD5, "results.json": N23_RESULTS_MD5}.items() if not (_ev / f).exists() or md5(_ev / f) != h]
_jf = N23 / "external_judgement.json"
G0 = {"a": {"ok": not _bad, "wrong_or_missing": _bad, "files_checked": len(SCORE_MD5) + 1,
            "drive_main_json_is_the_recorded_file": bool(_jf.exists() and md5(_jf) == N23_JSON_MD5)}}
print(f"G0(a) · notebook 23's {len(SCORE_MD5)} score files and its results file in Drive are the recorded ones:", not _bad,
      "· its main JSON is byte-identical too:", G0["a"]["drive_main_json_is_the_recorded_file"])
if _bad:
    raise SystemExit(f"stop: G0(a) failed. In Drive › P04_ExternalJudgement/dcase2020_evaluation these are missing or are not notebook 23's files: {_bad[:4]}")
DEV23, EV23 = N23_DEV, N23_EVAL                                       # notebook 23's numbers per type (AUC, pAUC), from its results zip

# G0(b) · the clip counts per type, set and machine equal notebook 23's (every zip's MD5 was checked by get_file)
G0["b"] = {"ok": bool(DCASE["counts"] == N23_COUNTS), "clips": {t: len(DMETA[t]["name"]) for t in TYPES}}
print("G0(b) · the clip counts per type, set and machine equal notebook 23's:", G0["b"]["ok"])
if not G0["b"]["ok"]:
    raise SystemExit("stop: G0(b) failed, this session's clip lists are not notebook 23's")

# the frozen system (notebook 23's algebra, unchanged): per encoder, the seeds' F unit-normalised, concatenated, ÷√seeds; ECHO
# concatenated, ÷√2; global LDN1 with the machine's normal training clips as references; the mean of the two encoders' scores
_missing = [f for f in [f"emb_{c}_DC_XC_s{s}_f0.npz" for _, c, sd in FROZEN for s in sd] + [f"dcase/echo_{t}.npy" for t in TYPES] if not (N23 / f).exists()]
if _missing:
    raise SystemExit(f"stop: notebook 23's embeddings are not in Drive › P04_ExternalJudgement (for example {_missing[0]})")
ECHO_DC = {t: np.load(N23 / "dcase" / f"echo_{t}.npy") for t in TYPES}
FM = {}


def frozen_matrix(code, seeds, t):
    k = (code, t)
    if k not in FM:
        Es = [np.load(N23 / f"emb_{code}_DC_XC_s{s}_f0.npz", allow_pickle=False)[f"F__{t}"].astype(np.float32) for s in seeds]
        E = np.concatenate([unit(e) for e in Es], 1) / math.sqrt(len(Es))
        FM[k] = np.concatenate([E, unit(ECHO_DC[t].astype(np.float64))], 1) / math.sqrt(2.0)
        assert len(FM[k]) == len(DMETA[t]["name"]), f"notebook 23's embeddings of {t} do not have this session's number of clips"
    return FM[k]


def machines(t, set_):
    return sorted(set(np.array(DMETA[t]["mid"])[dc_rows(t, set_, "test")].tolist()))


def frozen_score(t, ref, x):
    return sum(ldn1(frozen_matrix(c, sd, t)[ref], frozen_matrix(c, sd, t)[x]) for _, c, sd in FROZEN) / len(FROZEN)


def spread(logs):
    # the centre and the robust spread (1.4826 × the median absolute deviation) of a system's log scores on normal clips
    if len(logs) == 0:
        return {"n": 0, "median": None, "sigma": None}
    med = float(np.median(logs))
    return {"n": int(len(logs)), "median": med, "sigma": float(1.4826 * np.median(np.abs(logs - med)))}


t_g0 = time.time()
LAB = {t: np.array(DMETA[t]["label"]) for t in TYPES}
SF, CAL_F = {}, {}                                                    # (type, set, machine) → the frozen system's scores of its test clips
for t in TYPES:
    for set_ in ("dev", "eval"):
        for q in machines(t, set_):
            SF[(t, set_, q)] = frozen_score(t, dc_rows(t, set_, "train", q), dc_rows(t, set_, "test", q))
    # form Z's calibration of the frozen system (Amendment 1): its log score on the additional-training clips that its trainings did
    # not use (those after the first FROZEN_CAP of an evaluation machine, by name), against the clips they did use. No test clip.
    _nm, _logs = np.array(DMETA[t]["name"]), []
    for q in machines(t, "eval"):
        rows = dc_rows(t, "eval", "train", q); rows = rows[np.argsort(_nm[rows], kind="stable")]
        if len(rows) > FROZEN_CAP:
            _logs.append(np.log(np.maximum(frozen_score(t, rows[:FROZEN_CAP], rows[FROZEN_CAP:]), LDN_FLOOR)))
    CAL_F[t] = spread(np.concatenate(_logs) if _logs else np.zeros(0))
    FM.clear(); gc.collect()
print("form Z · the frozen system's spread on normal clips it did not train on (clips, σ of the log score):",
      {t: (CAL_F[t]["n"], None if CAL_F[t]["sigma"] is None else round(CAL_F[t]["sigma"], 4)) for t in TYPES})

# G0(c) · the frozen system comes back: the development metrics of all machines, and the saved evaluation scores of every clip.
# Reading the saved scores needs no label, and nothing is scored anew: it is the same system, and the numbers must be the same.
worst_dev, n_dev = 0.0, 0
for t in TYPES:
    for q in machines(t, "dev"):
        m, r = dc_metrics(LAB[t][dc_rows(t, "dev", "test", q)], SF[(t, "dev", q)]), N23_DEV_PM[t][q]
        assert (m["n_test"], m["n_anomalies"]) == (r[2], r[3]), (t, q, m, r)
        worst_dev = max(worst_dev, abs(m["AUC"] - r[0]), abs(m["pAUC"] - r[1])); n_dev += 1
worst_ev, n_ev = 0.0, 0
for t in TYPES:
    for q in machines(t, "eval"):
        te = dc_rows(t, "eval", "test", q); names = [DMETA[t]["name"][i] for i in te]
        saved = dict(ln.split(",") for ln in (N23 / "dcase2020_evaluation" / f"anomaly_score_{t}_{q}.csv").read_text().splitlines() if ln)
        assert sorted(names) == sorted(saved), f"notebook 23's score file of {t} {q} lists other clips"
        a = np.array([float(saved[nm]) for nm in names])
        worst_ev = max(worst_ev, float(np.max(np.abs(a - SF[(t, "eval", q)]) / np.maximum(np.abs(a), 1e-12)))); n_ev += len(te)
G0["c"] = {"ok": bool(worst_dev <= G0C_TOL and worst_ev <= G0C_REL), "development_machines": n_dev, "development_worst_abs_difference": worst_dev,
           "evaluation_clips": n_ev, "evaluation_worst_relative_difference": worst_ev, "tolerances": [G0C_TOL, G0C_REL]}
print(f"G0(c) · the frozen system recomputed from notebook 23's embeddings: development, {n_dev} machines, largest |Δ| {worst_dev:.1e} "
      f"(tolerance {G0C_TOL}); evaluation scores, {n_ev} clips, largest relative difference {worst_ev:.1e} (tolerance {G0C_REL}): {G0['c']['ok']}"
      f" · {time.time() - t_g0:.0f} s")
if not G0["c"]["ok"]:
    raise SystemExit("stop: G0(c) failed. Notebook 23's saved embeddings do not give its numbers with this session's clip order")

In [ ]:
# 5 · The second view: TASTgram-MFN trained with Noisy-ArcMix (Choi and Choi, ICASSP 2024, arXiv 2310.06364)
#
# The model classes, the log-mel settings and the Noisy-ArcMix step are adapted from https://github.com/soonhyeon/Noisy-ArcMix at
# commit 674c759bfaa7723619a4b1d8d00eb9d155c50ba6 (model/net.py, losses.py, dataloader.py, utils.py, trainer.py). Its model file
# says: "modification made on the basis of link:https://github.com/liuyoude/STgram-MFN". Its licence:
#
#   MIT License
#
#   Copyright (c) 2023 soonhyeon
#
#   Permission is hereby granted, free of charge, to any person obtaining a copy
#   of this software and associated documentation files (the "Software"), to deal
#   in the Software without restriction, including without limitation the rights
#   to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
#   copies of the Software, and to permit persons to whom the Software is
#   furnished to do so, subject to the following conditions:
#
#   The above copyright notice and this permission notice shall be included in all
#   copies or substantial portions of the Software.
#
#   THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
#   IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
#   FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
#   AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
#   LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
#   OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
#   SOFTWARE.
#
# Changed here (the rules list every deviation): the number of classes; the frame count and the last kernel follow the clip length
# (313 and 8 × 20 for DCASE's 10 s clips, as in the reference); mixed precision with a gradient scaler, and the ArcFace layer in
# float32 with a floor under its square root; batches come from one int16 tensor and the log-mel is computed on the device; the
# seeds, the hold-out draw, saving and resuming are this notebook's. The reference's unused options (sub-centres, easy margin,
# the other training modes) are left out.
class Bottleneck(nn.Module):
    def __init__(self, inp, oup, stride, expansion):
        super(Bottleneck, self).__init__()
        self.connect = stride == 1 and inp == oup
        self.conv = nn.Sequential(
            # pw
            nn.Conv2d(inp, inp * expansion, 1, 1, 0, bias=False),
            nn.BatchNorm2d(inp * expansion),
            nn.PReLU(inp * expansion),
            # dw
            nn.Conv2d(inp * expansion, inp * expansion, 3, stride, 1, groups=inp * expansion, bias=False),
            nn.BatchNorm2d(inp * expansion),
            nn.PReLU(inp * expansion),
            # pw-linear
            nn.Conv2d(inp * expansion, oup, 1, 1, 0, bias=False),
            nn.BatchNorm2d(oup),
        )

    def forward(self, x):
        if self.connect:
            return x + self.conv(x)
        else:
            return self.conv(x)


class ConvBlock(nn.Module):
    def __init__(self, inp, oup, k, s, p, dw=False, linear=False):
        super(ConvBlock, self).__init__()
        self.linear = linear
        if dw:
            self.conv = nn.Conv2d(inp, oup, k, s, p, groups=inp, bias=False)
        else:
            self.conv = nn.Conv2d(inp, oup, k, s, p, bias=False)
        self.bn = nn.BatchNorm2d(oup)
        if not linear:
            self.prelu = nn.PReLU(oup)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        if self.linear:
            return x
        else:
            return self.prelu(x)


Mobilefacenet_bottleneck_setting = [
    # t, c , n ,s
    [2, 128, 2, 2],
    [4, 128, 2, 2],
    [4, 128, 2, 2],
]


class MobileFaceNet(nn.Module):
    def __init__(self, num_class, last_kernel, bottleneck_setting=Mobilefacenet_bottleneck_setting):
        super(MobileFaceNet, self).__init__()
        self.conv1 = ConvBlock(3, 64, 3, 2, 1)
        self.dw_conv1 = ConvBlock(64, 64, 3, 1, 1, dw=True)
        self.inplanes = 64
        block = Bottleneck
        self.blocks = self._make_layer(block, bottleneck_setting)
        self.conv2 = ConvBlock(bottleneck_setting[-1][1], 512, 1, 1, 0)
        self.linear7 = ConvBlock(512, 512, last_kernel, 1, 0, dw=True, linear=True)   # (8, 20) in the reference: 128 × 313 → 8 × 20
        self.linear1 = ConvBlock(512, 128, 1, 1, 0, linear=True)
        self.fc_out = nn.Linear(128, num_class)                       # the reference has it too; with ArcFace on, its output is unused
        # init
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()

    def _make_layer(self, block, setting):
        layers = []
        for t, c, n, s in setting:
            for i in range(n):
                if i == 0:
                    layers.append(block(self.inplanes, c, s, t))
                else:
                    layers.append(block(self.inplanes, c, 1, t))
                self.inplanes = c
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.dw_conv1(x)
        x = self.blocks(x)
        x = self.conv2(x)
        x = self.linear7(x)
        x = self.linear1(x)
        return x.reshape(x.size(0), -1)                               # the 128-d feature


class TgramNet(nn.Module):
    def __init__(self, n_frames, num_layer=3, mel_bins=128, win_len=1024, hop_len=512):
        super(TgramNet, self).__init__()
        # if "center=True" of stft, padding = win_len / 2
        self.conv_extrctor = nn.Conv1d(1, mel_bins, win_len, hop_len, win_len // 2, bias=False)
        self.conv_encoder = nn.Sequential(
            *[nn.Sequential(
                nn.LayerNorm(n_frames),                               # 313 in the reference (10 s)
                nn.LeakyReLU(0.2, inplace=True),
                nn.Conv1d(mel_bins, mel_bins, 3, 1, 1, bias=False),
            ) for _ in range(num_layer)])

    def forward(self, x):
        out = self.conv_extrctor(x)
        out = self.conv_encoder(out)
        return out


class Temporal_Attention(nn.Module):
    def __init__(self, feature_dim=128):
        super().__init__()
        self.feature_dim = feature_dim
        self.max_pool = nn.AdaptiveMaxPool1d(1)
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: (B, 1, 128, 313)
        x = x.squeeze(1)
        x = x.transpose(1, 2)  # (B, 313, 128)
        x1 = self.max_pool(x)  # (B, 313, 1)
        x2 = self.avg_pool(x)  # (B, 313, 1)
        feats = x1 + x2
        feats = feats.repeat(1, 1, self.feature_dim)
        refined_feats = self.sigmoid(feats).transpose(1, 2) * x.transpose(1, 2)
        return refined_feats


class ArcMarginProduct(nn.Module):
    def __init__(self, in_features=128, out_features=200, s=30.0, m=0.7):
        super(ArcMarginProduct, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m
        self.weight = Parameter(torch.Tensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        # make the function cos(theta+m) monotonic decreasing while theta in [0°,180°]
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, x, label):
        cosine = F.linear(F.normalize(x), F.normalize(self.weight))
        sine = torch.sqrt((1.0 - torch.pow(cosine, 2)).clamp_min(SV["sine_floor"]))   # the floor is this notebook's (a deviation)
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where((cosine - self.th) > 0, phi, cosine - self.mm)
        one_hot = torch.zeros(cosine.size(), device=x.device)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1)
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output = output * self.s
        return output


MEMFMT = torch.contiguous_format                                      # set by pick_memory_format (a speed setting only)


class TASTgramMFN(nn.Module):
    def __init__(self, num_classes, n_frames, last_kernel, c_dim=128, win_len=1024, hop_len=512,
                 bottleneck_setting=Mobilefacenet_bottleneck_setting, m=0.7, s=30):
        super().__init__()
        self.arcface = ArcMarginProduct(in_features=c_dim, out_features=num_classes, m=m, s=s)
        self.tgramnet = TgramNet(n_frames, mel_bins=c_dim, win_len=win_len, hop_len=hop_len)
        self.mobilefacenet = MobileFaceNet(num_class=num_classes, last_kernel=last_kernel, bottleneck_setting=bottleneck_setting)
        self.temporal_attention = Temporal_Attention(feature_dim=c_dim)

    def forward(self, x_wav, x_mel, label):
        # the reference's "noisy_arcmix" mode: the margin goes to `label` (in training, the first clip's class)
        with torch.autocast(device_type=DEVICE, enabled=AMP):
            x_t = self.tgramnet(x_wav).unsqueeze(1)
            x_mel_temp_att = self.temporal_attention(x_mel).unsqueeze(1)
            x = torch.cat((x_t.float(), x_mel, x_mel_temp_att), dim=1)   # .float(): the three channels in one type under autocast
            feature = self.mobilefacenet(x.contiguous(memory_format=MEMFMT))
        feature = feature.float()
        with torch.autocast(device_type=DEVICE, enabled=False):      # ArcFace in float32 (a deviation)
            out = self.arcface(feature, label)
        return out, feature


MEL = torchaudio.transforms.MelSpectrogram(sample_rate=SR, n_fft=SV["n_fft"], hop_length=SV["hop"], n_mels=SV["n_mels"], power=2,
                                           pad_mode="constant", norm="slaney", mel_scale="slaney").to(DEVICE)
CE = nn.CrossEntropyLoss()


@torch.no_grad()
def log_mel(x):
    # (B, samples) float32 in [−1, 1] → (B, 1, mel bands, frames): the reference's file_to_log_mel_spectrogram with power = 2
    return (20.0 / 2 * torch.log10(MEL(x) + sys.float_info.epsilon)).unsqueeze(1)


def frames_of(ns):
    # the frames of an STFT with center=True, which is also the length of the Tgram convolution (kernel 1024, stride 512, padding 512)
    return 1 + ns // SV["hop"]


def last_kernel(n_mels, n_frames):
    # the feature map after the network's four stride-2 stages (3 × 3, padding 1): 128 × 313 → 8 × 20
    h, w = n_mels, n_frames
    for _ in range(4):
        h, w = (h - 1) // 2 + 1, (w - 1) // 2 + 1
    return (h, w)


def training_lists():
    # one class per machine: for each type, its development machines (all train/ clips of the development zip), then its evaluation
    # machines (all clips of the additional training zip); items (type index, row in the type's store, class)
    items, classes, cls_of = [], [], {}
    for ti, t in enumerate(TYPES):
        nm, mids = np.array(DMETA[t]["name"]), np.array(DMETA[t]["mid"])
        for set_, what in (("dev", "development"), ("eval", "additional")):
            for q in sorted(set(mids[dc_rows(t, set_, "train")].tolist())):
                rows = dc_rows(t, set_, "train", q); rows = rows[np.argsort(nm[rows], kind="stable")]
                c = len(classes); classes.append(f"{t}|{q}|DCASE2020-{what}"); cls_of[(t, q)] = c
                items.extend((ti, int(r), c) for r in rows)
    return np.array(items, dtype=np.int64), classes, cls_of


AUDIO, LABELS, STORE_DEV, MF_TIMES = None, None, None, {}


def build_audio():
    # every training-list clip in one int16 tensor: on the GPU when it has room (batches are then gathered without the CPU), else in RAM
    global AUDIO, LABELS, STORE_DEV
    if AUDIO is not None:
        return
    n = len(ITEMS); need = n * NS * 2 / 1e9; dev = DEVICE; t0 = time.time()
    if DEVICE == "cuda" and torch.cuda.mem_get_info()[0] / 1e9 < need + SV["gpu_headroom_gb"]:
        dev = "cpu"
    A = torch.empty((n, NS), dtype=torch.int16, device=dev)
    for ti, t in enumerate(TYPES):
        pos = np.where(ITEMS[:, 0] == ti)[0]
        for i in range(0, len(pos), 256):
            p = pos[i:i + 256]
            A[torch.from_numpy(p).to(dev)] = torch.from_numpy(np.ascontiguousarray(DST[t][ITEMS[p, 1]])).to(dev)
    AUDIO, STORE_DEV, LABELS = A, dev, torch.from_numpy(ITEMS[:, 2].copy()).to(dev)
    print(f"training audio: {n} clips, {need:.2f} GB as int16, kept on {'the GPU' if dev == 'cuda' else 'the CPU'} · {time.time() - t0:.0f} s", flush=True)


def fetch(ids):
    # clips `ids` of the training lists → float32 waves in [−1, 1] and their classes, on the device
    j = torch.from_numpy(np.ascontiguousarray(ids)).to(AUDIO.device)
    return AUDIO[j].to(DEVICE).float() / 32768.0, LABELS[j].to(DEVICE)


def build_net(seed):
    torch.manual_seed(seed)
    return TASTgramMFN(len(CLASSES), N_FRAMES, LAST_KERNEL, c_dim=SV["c_dim"], win_len=SV["win_len"], hop_len=SV["hop"], m=SV["arc_m"], s=SV["arc_s"])


def mixed_step(net, x, y, lam, perm):
    # the reference's mixup_data and noisy_arcmix_criterion: the waves and the log-mels are mixed with one λ and one permutation;
    # the margin is at the first clip's class; the loss is λ·CE(first class) + (1 − λ)·CE(second class) on the same logits
    mel = log_mel(x)
    logits, _ = net((lam * x + (1 - lam) * x[perm]).unsqueeze(1), lam * mel + (1 - lam) * mel[perm], y)
    return lam * CE(logits, y) + (1 - lam) * CE(logits, y[perm]), logits


def pick_memory_format():
    # the same mathematics in two memory layouts: a few training steps are timed in each on this GPU, and the faster one is kept.
    # These are also the first real training steps: if neither layout runs with the audio on the GPU (most likely the GPU's memory
    # is too small for both), the audio is moved to the CPU's memory and the steps are tried once more.
    global MEMFMT, AUDIO, LABELS, STORE_DEV
    if DEVICE != "cuda":
        return
    err = ""
    for attempt in (0, 1):
        MF_TIMES.clear()
        x, y = fetch(np.arange(min(SV["batch"], len(ITEMS)))); perm = torch.arange(len(y) - 1, -1, -1, device=DEVICE)
        for name, fmt in (("contiguous", torch.contiguous_format), ("channels_last", torch.channels_last)):
            MEMFMT = fmt; net = opt = scaler = loss = None
            try:
                net = build_net(0).to(DEVICE).to(memory_format=fmt); net.train()
                opt = torch.optim.AdamW(net.parameters(), lr=SV["lr"]); scaler = torch.amp.GradScaler("cuda", enabled=AMP)
                for i in range(16):
                    if i == 4:
                        torch.cuda.synchronize(); t0 = time.time()
                    loss, _ = mixed_step(net, x, y, 0.5, perm)
                    opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                torch.cuda.synchronize(); MF_TIMES[name] = round((time.time() - t0) / 12, 4)
            except Exception as e:
                MF_TIMES[name] = None; err = repr(e)[:160]
                print(f"  ({name} could not be timed: {err})", flush=True)
            del net, opt, scaler, loss
            gc.collect(); torch.cuda.empty_cache()
        ok = {k: v for k, v in MF_TIMES.items() if v}
        del x, y, perm
        if ok or STORE_DEV != "cuda":
            break
        AUDIO, LABELS, STORE_DEV = AUDIO.cpu(), LABELS.cpu(), "cpu"
        gc.collect(); torch.cuda.empty_cache()
        print("  no training step ran with the audio on the GPU: the audio is moved to the CPU's memory, and the steps are tried again", flush=True)
    if not ok:
        raise SystemExit(f"stop: a training step does not run on this GPU ({err}). Choose G4 in Runtime › Change runtime type, then Run all.")
    best = min(ok, key=ok.get)
    MEMFMT = torch.channels_last if best == "channels_last" else torch.contiguous_format
    print(f"memory layout: seconds per training step {MF_TIMES} → {best}", flush=True)


@torch.no_grad()
def hold_out_pass(net, hold):
    # as the reference's valid(): evaluation mode, margin at the clip's class, no mixing; the mean of the batches' means
    net.eval(); losses, accs = [], []
    for i in range(0, len(hold), SV["batch"]):
        x, y = fetch(hold[i:i + SV["batch"]])
        logits, _ = net(x.unsqueeze(1), log_mel(x), y)
        losses.append(float(CE(logits, y))); accs.append(float((logits.argmax(1) == y).float().mean()))
    return float(np.mean(losses)), float(np.mean(accs))


def fmt_time(s):
    return f"{s / 3600:.1f} h" if s >= 5400 else f"{s / 60:.0f} min"


def hold_out_split():
    # 5 % held out at random, one draw for the three seeds (Amendment 1): the held-out clips are then unseen by every seed's model.
    # The draw is saved in Drive the first time and read from there afterwards, so that a later session uses the same clips.
    n = len(ITEMS); n_tr = int(n * SV["split_ratio"]); f = OUT / "hold_out.json"
    if f.exists():
        d = json.loads(f.read_text())
        assert d["n_clips"] == n and len(d["hold_out"]) == n - n_tr, "the saved hold-out belongs to other training lists"
        hold = np.array(d["hold_out"], dtype=np.int64)
    else:
        hold = np.sort(np.random.default_rng([25, 0]).permutation(n)[n_tr:])
        f.write_text(json.dumps({"n_clips": int(n), "hold_out": hold.tolist()}))
    return np.setdiff1d(np.arange(n), hold), hold


def train_sv(seed):
    # one training of the second view: 5 % held out, Noisy-ArcMix on every batch, AdamW with cosine annealing per epoch, the model
    # with the lowest hold-out loss kept (as the reference); saved to Drive every SV["save_every"] epochs and resumed from there
    fin, res = OUT / f"sv_s{seed}.pt", OUT / f"sv_s{seed}_resume.pt"
    build_audio()
    tr_idx, hold = hold_out_split(); n, n_tr = len(ITEMS), len(tr_idx)
    hold_md5 = hashlib.md5(hold.astype("<i8").tobytes()).hexdigest()
    bs, epochs = SV["batch"], SV["epochs"]; steps = math.ceil(n_tr / bs)
    net = build_net(seed).to(DEVICE).to(memory_format=MEMFMT)
    opt = torch.optim.AdamW(net.parameters(), lr=SV["lr"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=SV["eta_min_ratio"] * SV["lr"])
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    log = {"seed": seed, "cfg": dict(SV), "classes": CLASSES, "n_clips": int(n), "n_train": int(n_tr), "n_hold_out": int(n - n_tr),
           "hold_out_md5": hold_md5, "steps_per_epoch": int(steps), "n_frames": int(N_FRAMES), "last_kernel": list(LAST_KERNEL),
           "parameters": int(sum(p.numel() for p in net.parameters())), "epochs": [], "resumed_at_epoch": [], "sessions": []}
    best, start = {"val_loss": float("inf"), "epoch": -1, "state": None}, 0
    if res.exists():
        st = torch.load(res, map_location="cpu", weights_only=True)
        log = json.loads(st["log"])
        assert log["classes"] == CLASSES and log["hold_out_md5"] == hold_md5 and log["cfg"] == json.loads(json.dumps(SV)), \
            "the saved training used other lists or settings"
        net.load_state_dict(st["model"]); opt.load_state_dict(st["opt"]); sched.load_state_dict(st["sched"]); scaler.load_state_dict(st["scaler"])
        best = {"val_loss": st["best_val_loss"], "epoch": st["best_epoch"], "state": st["best_state"]}
        start = st["epoch"] + 1; log["resumed_at_epoch"].append(start)
        print(f"seed {seed}: resuming after epoch {start}", flush=True)
    session = {"gpu": GPU, "memory_layout": "channels_last" if MEMFMT == torch.channels_last else "contiguous", "audio_on": STORE_DEV,
               "from_epoch": start, "step_seconds": None}
    log["sessions"].append(session)
    print(f"seed {seed}: {n_tr} training clips and {n - n_tr} held out · {len(CLASSES)} classes · {steps} steps per epoch · "
          f"{log['parameters']:,} parameters", flush=True)
    t_from, t_to = min(5, steps - 2), min(45, steps - 1)
    for ep in range(start, epochs):
        rng = np.random.default_rng([seed, 25, 1, ep]); order = tr_idx[rng.permutation(n_tr)]
        lr_now = opt.param_groups[0]["lr"]
        net.train(); t0, tot, acc = time.time(), 0.0, 0.0
        for b in range(steps):
            x, y = fetch(order[b * bs:(b + 1) * bs])
            lam = float(rng.beta(SV["alpha"], SV["alpha"])); pb = torch.from_numpy(rng.permutation(len(y))).to(DEVICE)
            loss, logits = mixed_step(net, x, y, lam, pb)
            if not bool(torch.isfinite(loss)):
                raise SystemExit(f"stop: G1 failed. The training loss of seed {seed} is not finite (epoch {ep + 1}, step {b + 1})")
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            tot += float(loss.detach()); acc += float((logits.detach().argmax(1) == y).float().mean())
            if ep == start and b in (t_from, t_to) and t_to > t_from >= 0:
                if DEVICE == "cuda":
                    torch.cuda.synchronize()
                if b == t_from:
                    t_mark = time.time()
                else:
                    session["step_seconds"] = round((time.time() - t_mark) / (t_to - t_from), 4)
                    left_here = (epochs - ep) * steps * session["step_seconds"] * 1.03
                    later = [s for s in SV["seeds"] if s != seed and not (OUT / f"sv_s{s}.pt").exists()]
                    left_all = left_here + len(later) * epochs * steps * session["step_seconds"] * 1.03
                    print(f"  this GPU ({GPU or 'CPU'}): {session['step_seconds']:.3f} s per step → about {fmt_time(left_here)} left for this "
                          f"training, about {fmt_time(left_all)} for all the trainings that are left", flush=True)
                    if left_all > 6 * 3600 and not TEST:
                        print("  NOTE: that is a long run on this GPU. It can stay as it is (it resumes after a disconnect). For a much shorter "
                              "run: Runtime › Change runtime type › G4, then Runtime › Run all; what is saved in Drive is kept.", flush=True)
        sched.step()
        vl, va = hold_out_pass(net, hold)
        if not math.isfinite(vl):
            raise SystemExit(f"stop: G1 failed. The hold-out loss of seed {seed} is not finite (epoch {ep + 1})")
        if best["val_loss"] > vl:                                     # the reference saves the model whenever the hold-out loss falls
            best = {"val_loss": vl, "epoch": ep, "state": {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}}
        log["epochs"].append({"epoch": ep, "train_loss": round(tot / steps, 5), "train_accuracy": round(acc / steps, 5), "hold_out_loss": round(vl, 5),
                              "hold_out_accuracy": round(va, 5), "lr": lr_now, "seconds": round(time.time() - t0, 1)})
        if ep < start + 3 or (ep + 1) % 5 == 0 or ep + 1 == epochs:
            print(f"seed {seed} · epoch {ep + 1}/{epochs} · loss {tot / steps:.4f} · hold-out loss {vl:.4f}, accuracy {va:.4f} · "
                  f"best epoch {best['epoch'] + 1} · {time.time() - t0:.0f} s", flush=True)
        if (ep + 1) % SV["save_every"] == 0 and ep + 1 < epochs:
            tmp = res.with_name(res.name + ".tmp")
            torch.save({"model": {k: v.detach().cpu() for k, v in net.state_dict().items()}, "opt": opt.state_dict(), "sched": sched.state_dict(),
                        "scaler": scaler.state_dict(), "epoch": ep, "best_state": best["state"], "best_epoch": best["epoch"],
                        "best_val_loss": best["val_loss"], "log": json.dumps(log)}, tmp)
            os.replace(tmp, res)
    log["best_epoch"], log["best_hold_out_loss"] = int(best["epoch"]), float(best["val_loss"])
    tmp = fin.with_name(fin.name + ".tmp")
    torch.save({"state": best["state"], "last_state": {k: v.detach().cpu() for k, v in net.state_dict().items()}, "best_epoch": int(best["epoch"]),
                "best_val_loss": float(best["val_loss"]), "classes": CLASSES, "hold_out": hold.tolist(), "log": json.dumps(log)}, tmp)
    os.replace(tmp, fin)
    res.unlink(missing_ok=True)
    del net, opt, sched, scaler
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


@torch.no_grad()
def embed_sv(seed):
    # the saved model on every DCASE clip: the 128-d feature, and the paper's score (cross-entropy for the clip's own machine class)
    fin = OUT / f"sv_s{seed}.pt"
    st = torch.load(fin, map_location="cpu", weights_only=True)
    assert st["classes"] == CLASSES, "the saved model used other classes"
    net = build_net(seed); net.load_state_dict(st["state"]); net.to(DEVICE).to(memory_format=MEMFMT).eval()
    out, bs = {}, SV["embed_batch"]
    for t in TYPES:
        y_all = np.array([CLS_OF[(t, q)] for q in DMETA[t]["mid"]], dtype=np.int64); n = DST[t].shape[0]
        V, ce = np.zeros((n, SV["c_dim"]), np.float32), np.zeros(n, np.float32)
        for i in range(0, n, bs):
            x = torch.from_numpy(f32(DST[t][i:i + bs])).to(DEVICE); y = torch.from_numpy(y_all[i:i + bs]).to(DEVICE)
            logits, feat = net(x.unsqueeze(1), log_mel(x), y)
            V[i:i + len(y)] = feat.cpu().numpy(); ce[i:i + len(y)] = F.cross_entropy(logits, y, reduction="none").cpu().numpy()
        assert np.isfinite(V).all() and np.isfinite(ce).all(), f"non-finite features for {t}"
        out[f"V__{t}"], out[f"CE__{t}"] = V, ce
    tmp = OUT / f"emb_SV_s{seed}.part.npz"
    np.savez(tmp, **out, meta=json.dumps({"seed": seed, "weights_md5": md5(fin), "best_epoch": int(st["best_epoch"]), "test_run": TEST,
                                          "date": time.strftime("%Y-%m-%d %H:%M")}))
    os.replace(tmp, OUT / f"emb_SV_s{seed}.npz")
    del net
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

In [ ]:
# 6 · How systems are scored on a machine's test clips, and the development tables
VIEWS, CES, VM = {}, {}, {}                                           # seed → type → features (n, 128) and the paper's score (n,)
HOLD_ROWS, CAL_V, CAL_P = {}, {}, {}                                  # type → store rows of the held-out clips; (seeds, type) → the view's spreads
SYSTEMS = ["frozen", "view_LDN1", "view_B1", "view_paper"] + FORMS
LABEL = {"frozen": "frozen (notebook 23)", "view_LDN1": "view alone · LDN1", "view_B1": "view alone · B1", "view_paper": "view alone · paper's score",
         "A": "A · mean of scores", "G": "G · geometric mean", "C": "C · concatenation", "Z": "Z · calibrated sum (LDN1)",
         "ZP": "ZP · calibrated sum (paper)"}
NAME = {"pump": "pump", "fan": "fan", "valve": "valve", "slider": "slide rail"}


def view_matrix(seeds, t):
    # the program's seed ensemble: each seed's features unit-normalised, concatenated, ÷√seeds
    k = (tuple(seeds), t)
    if k not in VM:
        VM[k] = np.concatenate([unit(VIEWS[s][t]) for s in seeds], 1) / math.sqrt(len(seeds))
    return VM[k]


def ldn1_left_out(R, pos):
    # LDN1 of the reference clips at positions `pos`, each against all the other references (the clip itself left out)
    U = unit(R); S = U @ U.T; np.fill_diagonal(S, -np.inf); out = []
    for h in pos:
        keep = np.ones(len(U), bool); keep[h] = False
        d1 = np.maximum(1.0 - S[np.ix_(keep, keep)].max(1), LDN_FLOOR)
        out.append(float(((1.0 - S[h, keep]) / d1).min()))
    return np.array(out)


def calibrate_view(seeds, t):
    # form Z's calibration of the view (Amendment 1): its log score on the held-out clips, which no seed trained on, each against all
    # the other training clips of its machine (the references a test clip meets, less the clip itself). No test clip.
    k = (tuple(seeds), t)
    if k not in CAL_V:
        V, logs = view_matrix(seeds, t), []
        for set_ in ("dev", "eval"):
            for q in machines(t, set_):
                rows = dc_rows(t, set_, "train", q); pos = np.where(np.isin(rows, HOLD_ROWS[t]))[0]
                if len(pos) and len(rows) >= 3:
                    logs.append(np.log(np.maximum(ldn1_left_out(V[rows], pos), LDN_FLOOR)))
        CAL_V[k] = spread(np.concatenate(logs) if logs else np.zeros(0))
    return CAL_V[k]


_R = np.random.default_rng(5).standard_normal((30, 12))
assert np.allclose(ldn1_left_out(_R, [3, 17]), [ldn1(np.delete(_R, h, 0), _R[[h]])[0] for h in (3, 17)]), "stop: the left-out LDN1 is not LDN1"


def calibrate_paper(seeds, t):
    # form ZP's calibration of the view: the log of the paper's score (averaged over the seeds) on the held-out clips. No test clip.
    k = (tuple(seeds), t)
    if k not in CAL_P:
        rows = np.asarray(HOLD_ROWS[t])
        CAL_P[k] = spread(np.log(np.maximum(sum(CES[s][t][rows] for s in seeds) / len(seeds), LDN_FLOOR)) if len(rows) else np.zeros(0))
    return CAL_P[k]


def z_ok(seeds, name="Z"):
    # a calibrated form needs a spread from at least CAL_MIN clips, for both systems and every type
    cal = calibrate_view if name == "Z" else calibrate_paper
    return all(c["n"] >= CAL_MIN and c["sigma"] is not None and c["sigma"] > 1e-9 for t in TYPES for c in (CAL_F[t], cal(seeds, t)))


def score_of(name, t, set_, q, seeds):
    # one system's anomaly scores of one machine's test clips; the references are the machine's normal training clips (as notebook 23)
    ref, te = dc_rows(t, set_, "train", q), dc_rows(t, set_, "test", q)
    if name == "frozen":
        return SF[(t, set_, q)]
    if name == "view_paper":                                          # the paper's score, averaged over the seeds
        return sum(CES[s][t][te] for s in seeds) / len(seeds)
    if name == "ZP":                                                  # as Z, with the paper's score for the view
        return (np.log(np.maximum(SF[(t, set_, q)], LDN_FLOOR)) / CAL_F[t]["sigma"]
                + np.log(np.maximum(sum(CES[s][t][te] for s in seeds) / len(seeds), LDN_FLOOR)) / calibrate_paper(seeds, t)["sigma"])
    V = view_matrix(seeds, t)
    if name == "view_LDN1":
        return ldn1(V[ref], V[te])
    if name == "view_B1":
        return b1(V[ref], V[te])
    if name == "A":                                                   # the mean of the two families' scores
        return 0.5 * SF[(t, set_, q)] + 0.5 * ldn1(V[ref], V[te])
    if name == "G":                                                   # their geometric mean
        return np.sqrt(np.maximum(SF[(t, set_, q)], LDN_FLOOR) * np.maximum(ldn1(V[ref], V[te]), LDN_FLOOR))
    if name == "Z":                                                   # each system's log score in units of its spread on unseen normal clips
        return (np.log(np.maximum(SF[(t, set_, q)], LDN_FLOOR)) / CAL_F[t]["sigma"]
                + np.log(np.maximum(ldn1(V[ref], V[te]), LDN_FLOOR)) / calibrate_view(seeds, t)["sigma"])
    if name == "C":                                                   # the view concatenated to each encoder's matrix, as ECHO was
        return sum(ldn1(np.concatenate([frozen_matrix(c, sd, t)[ref], V[ref]], 1) / math.sqrt(2.0),
                        np.concatenate([frozen_matrix(c, sd, t)[te], V[te]], 1) / math.sqrt(2.0)) for _, c, sd in FROZEN) / len(FROZEN)
    raise ValueError(name)


def dev_eval(seeds, names, keep=None):
    # the development set. Per system and type: AUC and pAUC per machine and their means (the official metric), the worst machine,
    # and the AUC with the type's machines pooled (what the reference's script computes). With `keep`, every clip's scores are kept too.
    names = [nm for nm in names if nm not in ("Z", "ZP") or z_ok(seeds, nm)]   # a calibrated form only when it can be calibrated
    out = {nm: {} for nm in names}
    for t in TYPES:
        qs = machines(t, "dev"); ys = {q: LAB[t][dc_rows(t, "dev", "test", q)] for q in qs}
        for nm in names:
            sc = {q: score_of(nm, t, "dev", q, seeds) for q in qs}
            if keep is not None:
                for q in qs:
                    keep.setdefault((t, q), {"file": [DMETA[t]["name"][i] for i in dc_rows(t, "dev", "test", q)], "label": ys[q]})[nm] = sc[q]
            per = {q: dc_metrics(ys[q], sc[q]) for q in qs}
            y_all, s_all = np.concatenate([ys[q] for q in qs]), np.concatenate([sc[q] for q in qs])
            out[nm][t] = {"per_machine": per, "AUC": float(np.mean([v["AUC"] for v in per.values()])),
                          "pAUC": float(np.mean([v["pAUC"] for v in per.values()])),
                          "worst_machine_AUC": float(min(v["AUC"] for v in per.values())),
                          "pooled_AUC": float(roc_auc_score(y_all, s_all)), "pooled_pAUC": float(roc_auc_score(y_all, s_all, max_fpr=0.1))}
        FM.clear(); VM.clear(); gc.collect()
    return out


def s2(res):
    # the rules' two scores, four decimals: development = mean AUC of pumps and fans; confirmation = of valves and slide rails
    return (round(float(np.mean([res[t]["AUC"] for t in MAIN])), 4), round(float(np.mean([res[t]["AUC"] for t in AUX])), 4))


def g_view(res):
    # gate G-VIEW (Amendment 1): for every type, the view alone reaches the DCASE 2020 baseline on the development set with LDN1 or with
    # the paper's score. It is there to catch a broken training or a broken pass.
    return {t: {"AUC_LDN1": round(res["view_LDN1"][t]["AUC"], 4), "AUC_paper": round(res["view_paper"][t]["AUC"], 4), "baseline": DCASE_BASELINE_DEV[t],
                "ok": bool(max(res["view_LDN1"][t]["AUC"], res["view_paper"][t]["AUC"]) >= DCASE_BASELINE_DEV[t])} for t in TYPES}


def show(res, names, title, key="AUC"):
    print(title)
    print(f"  {'':<32}" + "".join(f"{NAME[t]:>12}" for t in TYPES) + (f"{'development':>14}{'confirmation':>14}" if key == "AUC" else ""))
    for nm in [n_ for n_ in names if n_ in res]:
        d, c = s2(res[nm])
        print(f"  {LABEL[nm]:<32}" + "".join(f"{res[nm][t][key]:>12.4f}" for t in TYPES) + (f"{d:>14.4f}{c:>14.4f}" if key == "AUC" else ""))

In [ ]:
# 7 · Train the three seeds (each: training or resume, then its features of every DCASE clip); a preview after the first seed
ITEMS, CLASSES, CLS_OF = training_lists()
N_FRAMES = frames_of(NS); LAST_KERNEL = last_kernel(SV["n_mels"], N_FRAMES)
_, HOLD = hold_out_split(); N_TRAIN = len(ITEMS) - len(HOLD)
HOLD_ROWS.update({t: ITEMS[HOLD][ITEMS[HOLD][:, 0] == ti][:, 1] for ti, t in enumerate(TYPES)})
print(f"training lists: {len(ITEMS)} clips ({int(sum(1 for c in CLASSES if c.endswith('additional')))} additional-training machines among "
      f"{len(CLASSES)} classes) · {N_FRAMES} frames · last kernel {LAST_KERNEL} · {N_TRAIN} train, {len(HOLD)} held out (the same clips for every seed)")
if not TEST:
    assert (N_FRAMES, LAST_KERNEL, len(CLASSES), len(ITEMS), N_TRAIN) == (REF_FRAMES, REF_KERNEL, REF_CLASSES, REF_CLIPS, REF_TRAIN), \
        "stop: the lists or the sizes are not the rules'"
_todo = [s for s in SV["seeds"] if not (OUT / f"sv_s{s}.pt").exists()]
print(f"trainings to do: {len(_todo)} of {len(SV['seeds'])} · {math.ceil(N_TRAIN / SV['batch'])} steps per epoch, {SV['epochs']} epochs each. "
      "My estimate before any run (not measured): about 1–3 h in all on an RTX PRO 6000 (G4), about 6–24 h on a T4. The measured estimate follows.")
if _todo:
    build_audio(); pick_memory_format()
LOGS, G1, T_TRAIN, T_EMBED, PREVIEW, GV0 = {}, {}, {}, {}, None, None


def ensure_sv(seed):
    # one seed: trained (or resumed, or read from Drive), then its features of every DCASE clip; gate G1
    fin, ef = OUT / f"sv_s{seed}.pt", OUT / f"emb_SV_s{seed}.npz"
    if not fin.exists():
        t0 = time.time(); train_sv(seed); T_TRAIN[seed] = round(time.time() - t0, 1)
    if ef.exists() and json.loads(str(np.load(ef, allow_pickle=False)["meta"]))["weights_md5"] != md5(fin):
        ef.unlink()                                                   # features of another model file: computed again
    if not ef.exists():
        t0 = time.time(); embed_sv(seed); T_EMBED[seed] = round(time.time() - t0, 1)
        print(f"seed {seed}: features of every DCASE clip saved · {T_EMBED[seed]:.0f} s", flush=True)
    D = np.load(ef, allow_pickle=False)
    VIEWS[seed] = {t: D[f"V__{t}"].astype(np.float64) for t in TYPES}; CES[seed] = {t: D[f"CE__{t}"].astype(np.float64) for t in TYPES}
    assert all(len(VIEWS[seed][t]) == len(DMETA[t]["name"]) for t in TYPES), "the saved features do not have this session's number of clips"
    st = torch.load(fin, map_location="cpu", weights_only=True)
    assert st["hold_out"] == HOLD.tolist(), "the saved model was trained with another hold-out"
    log = json.loads(st["log"]); LOGS[seed] = log
    e = log["epochs"]
    G1[seed] = {"first_train_loss": e[0]["train_loss"], "last_train_loss": e[-1]["train_loss"], "first_hold_out_loss": e[0]["hold_out_loss"],
                "best_hold_out_loss": round(log["best_hold_out_loss"], 5), "best_epoch": log["best_epoch"] + 1,
                "hold_out_accuracy_at_best": e[log["best_epoch"]]["hold_out_accuracy"], "epochs": len(e),
                "ok": bool(len(e) == SV["epochs"] and e[-1]["train_loss"] < e[0]["train_loss"] and log["best_hold_out_loss"] < e[0]["hold_out_loss"])}
    print(f"G1 · seed {seed}: training loss {e[0]['train_loss']:.4f} → {e[-1]['train_loss']:.4f}; hold-out loss {e[0]['hold_out_loss']:.4f} → "
          f"{log['best_hold_out_loss']:.4f} at epoch {log['best_epoch'] + 1} (accuracy {G1[seed]['hold_out_accuracy_at_best']:.4f}): {G1[seed]['ok']}", flush=True)
    if not G1[seed]["ok"] and not TEST:
        raise SystemExit(f"stop: G1 failed for seed {seed}")


for _i, _seed in enumerate(SV["seeds"]):
    ensure_sv(_seed)
    if _i == 0:
        PREVIEW = dev_eval([_seed], ["frozen", "view_LDN1", "view_paper"] + FORMS)
        GV0 = g_view(PREVIEW)
        show(PREVIEW, ["frozen", "view_LDN1", "view_paper"] + FORMS,
             f"PREVIEW with seed {_seed} alone (development AUC; not a decision, the other seeds are trained whatever it shows):")
        if "Z" not in PREVIEW or "ZP" not in PREVIEW:
            print("  (a calibrated form could not be calibrated with this seed alone)")
        print("G-VIEW after the first seed · the view alone reaches the DCASE 2020 baseline for every type (by LDN1 or by the paper's score):",
              all(v["ok"] for v in GV0.values()), GV0, flush=True)
        if not TEST and not all(v["ok"] for v in GV0.values()):
            raise SystemExit("stop: G-VIEW failed after the first seed. Something is broken in the view; nothing is decided")
AUDIO = LABELS = None
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
print(f"the trainings are done · {(time.time() - T_START) / 60:.0f} min in this session")

In [ ]:
# 8 · The decision on the DCASE 2020 development set (the rules: select the form on pumps and fans, then the program's keep rule)
SEEDS = list(SV["seeds"])
DEV_SC = {}                                                           # (type, machine) → every development test clip's scores, saved in the zip
DEV = dev_eval(SEEDS, SYSTEMS, keep=DEV_SC)
SYSTEMS = [nm for nm in SYSTEMS if nm in DEV]; FAMILY = [f for f in FORMS if f in DEV]
SC = {nm: s2(DEV[nm]) for nm in SYSTEMS}
CAL = {"frozen": CAL_F, "view": {t: calibrate_view(SEEDS, t) for t in TYPES}, "view_paper": {t: calibrate_paper(SEEDS, t) for t in TYPES},
       "frozen_cap": FROZEN_CAP, "minimum_clips": CAL_MIN, "Z_available": "Z" in FAMILY, "ZP_available": "ZP" in FAMILY}
_cs = lambda d: {t: (d[t]["n"], None if d[t]["sigma"] is None else round(d[t]["sigma"], 4)) for t in TYPES}
print("forms Z and ZP · spreads of the log score on normal training clips a system did not train on (clips, σ): frozen", _cs(CAL_F),
      "· view by LDN1", _cs(CAL["view"]), "· view by the paper's score", _cs(CAL["view_paper"]))
for _f in ("Z", "ZP"):
    if _f not in FAMILY:
        print(f"NOTE: form {_f} is left out: a spread rests on fewer than {CAL_MIN} clips (or is zero) for at least one type")
GV = g_view(DEV)
print(f"G-VIEW · the view alone, {len(SEEDS)} seeds, reaches the DCASE 2020 baseline for every type (by LDN1 or by the paper's score):",
      all(v["ok"] for v in GV.values()), GV)
if not TEST and not all(v["ok"] for v in GV.values()):
    raise SystemExit("stop: G-VIEW failed. Something is broken in the view; nothing is decided")
CUR = SC["frozen"]                                                   # this session's recomputation (gate G0(c) tied it to notebook 23's)
THR = KEEP_AT if not TEST else (round(CUR[0] + BAR, 4), round(CUR[1] - BAR, 4))   # the rules' two bars (the synthetic test derives its own)
if not TEST and CUR != CURRENT:
    print(f"NOTE: the frozen system's scores recomputed here are {CUR}; the rules quote {CURRENT}. The rules' bars {KEEP_AT} are used.")
BEST = max(FAMILY, key=lambda f: (SC[f][0], -FAMILY.index(f)))       # the highest development score; a tie goes to the earlier form
KEPT = bool(SC[BEST][0] >= THR[0] and SC[BEST][1] >= THR[1])
FORCED = False
if TEST and FORCE in ("keep", "drop"):
    KEPT, FORCED = FORCE == "keep", True
show(DEV, SYSTEMS, f"\nDevelopment set · AUC per type (mean over a type's four machines), the view with {len(SEEDS)} seeds:")
show(DEV, SYSTEMS, "\nDevelopment set · pAUC (p = 0.1):", key="pAUC")
print("\nThe paper's score with a type's machines pooled, as the reference's script computes it (AUC):",
      {NAME[t]: round(DEV["view_paper"][t]["pooled_AUC"], 4) for t in TYPES},
      "· published, 41 classes:", {NAME[t]: PUBLISHED_DEV["Noisy-ArcMix, TASTgram-MFN (arXiv 2310.06364)"][t][0] / 100 for t in TYPES})
print("The worst machine of each type (AUC):", {nm: {NAME[t]: round(DEV[nm][t]["worst_machine_AUC"], 4) for t in TYPES} for nm in ("frozen", "view_LDN1", BEST)})
PER_SEED = {s: dev_eval([s], ["view_LDN1", "view_B1", "view_paper", BEST]) for s in SEEDS}
print(f"\nSeed spread · form {BEST} with each seed alone (development, confirmation):", {s: s2(PER_SEED[s][BEST]) for s in SEEDS if BEST in PER_SEED[s]},
      "· the view alone (LDN1):", {s: s2(PER_SEED[s]["view_LDN1"]) for s in SEEDS})
DECISION = {"current": list(CUR), "keep_at": list(THR), "family": FAMILY, "forms": {f: list(SC[f]) for f in FAMILY}, "selected": BEST, "kept": KEPT,
            "passes_development": bool(SC[BEST][0] >= THR[0]), "passes_confirmation": bool(SC[BEST][1] >= THR[1]), "test_forced": FORCED}
print(f"\nDECISION · selected form {BEST} ({LABEL[BEST]}): development {SC[BEST][0]:.4f} against {THR[0]:.4f} needed "
      f"(now {CUR[0]:.4f}); confirmation {SC[BEST][1]:.4f} against {THR[1]:.4f} needed (now {CUR[1]:.4f}) → "
      + ("KEPT" if KEPT else "NOT KEPT") + (" (forced by the test)" if FORCED else ""))

In [ ]:
# 9 · The evaluation set: scored only if the keep rule passed, once, by the kept system. The scores are written before any label is
# read; a later run reads the saved results. No other system (the view alone, the other forms, single seeds) is scored on it.
EV_DIR = OUT / "dcase2020_evaluation_second_view"
EV_RES, EV_LOCK = EV_DIR / "results.json", EV_DIR / "scores_written.json"


def official_check():
    out = {"ran": False, "evaluator": {"url": EVALUATOR[0], "sha256": EVALUATOR[2]}, "labels": list(DC_GT[1:])}
    src = EV_DIR
    files = json.loads((src / "scores_written.json").read_text())["files"]
    if any(not (src / f).exists() for f in files) or not (src / "results.json").exists():
        out["reason"] = "a score file or results.json of this notebook is missing"
        return out
    wd = WORK / "official_evaluator"
    shutil.rmtree(wd, ignore_errors=True); (wd / "teams" / "P04").mkdir(parents=True)
    try:
        download(EVALUATOR[0], wd / EVALUATOR[1])
    except Exception as e:
        out["reason"] = f"the evaluator did not download ({e.__class__.__name__})"
        return out
    got = sha256(wd / EVALUATOR[1])
    if got != EVALUATOR[2]:
        out["reason"] = f"the evaluator is not the recorded file (its SHA-256 starts {got[:16]})"
        return out
    try:
        gt = get_file(DC_GT)                                          # stops unless the file's MD5 is the Zenodo record's
    except (SystemExit, Exception) as e:
        out["reason"] = f"the label file is not the recorded one, or it did not download ({str(e)[:120]})"
        return out
    # the evaluator stops on a machine type without score files (ToyCar and ToyConveyor were not scored), so it gets the label
    # file's sections of the four MIMII types only, with their rows unchanged
    kept, types_in_file, cur = [], [], None
    with open(gt, newline="") as fh:
        for line in fh:
            row = next(csv.reader([line]), [])
            if not row:
                continue
            if len(row) < 2:                                          # a machine-type line, as the evaluator reads it
                cur = row[0].strip(); types_in_file.append(cur)
            if cur in TYPES:
                kept.append(line)
    with open(wd / "eval_data_list.csv", "w", newline="") as fh:
        fh.writelines(kept)
    for f in files:
        shutil.copy(src / f, wd / "teams" / "P04" / f)
    r = subprocess.run([sys.executable, EVALUATOR[1]], cwd=wd, capture_output=True, text=True)   # the official script, unmodified
    res_f = wd / "teams" / "result_P04.csv"
    if r.returncode != 0 or not res_f.exists():
        out["reason"] = f"the evaluator stopped (code {r.returncode}): {(r.stderr or r.stdout)[-300:]}"
        return out
    off, cur = {}, None
    with open(res_f, newline="") as fh:
        for row in csv.reader(fh):
            if len(row) == 1:
                cur = row[0]; off[cur] = {"per_machine": {}}
            elif len(row) == 3 and row[0] == "Average":
                off[cur]["AUC"], off[cur]["pAUC"] = float(row[1]), float(row[2])
            elif len(row) == 3 and row[0] != "id":
                off[cur]["per_machine"]["id_" + row[0]] = {"AUC": float(row[1]), "pAUC": float(row[2])}
    ours = json.loads((src / "results.json").read_text())["per_type"]
    if set(off) != set(TYPES) or any(set(off[t]["per_machine"]) != set(ours[t]["per_machine"]) or "AUC" not in off[t] for t in TYPES):
        out["reason"] = "the evaluator's result file does not list the same machine types and machines"
        return out
    diffs = [abs(off[t][m] - ours[t][m]) for t in TYPES for m in ("AUC", "pAUC")]
    diffs += [abs(off[t]["per_machine"][q][m] - ours[t]["per_machine"][q][m]) for t in TYPES for q in ours[t]["per_machine"] for m in ("AUC", "pAUC")]
    m_auc, m_pauc = float(np.mean([off[t]["AUC"] for t in TYPES])), float(np.mean([off[t]["pAUC"] for t in TYPES]))
    out.update(ran=True, agree=bool(max(diffs) <= EVAL_TOL), max_abs_difference=float(max(diffs)), values_compared=len(diffs),
               machine_types_in_the_label_file=types_in_file, label_rows_given=len(kept), per_type=off,
               this_notebook={t: {"AUC": ours[t]["AUC"], "pAUC": ours[t]["pAUC"]} for t in TYPES},
               mean_official={"mean_AUC": round(m_auc, 4), "mean_pAUC": round(m_pauc, 4)})
    shutil.copy(res_f, EV_DIR / "official_evaluator_result.csv")
    return out


OFFICIAL = {"ran": False, "reason": "the evaluation set was not scored"}
if not KEPT and not (EV_RES.exists() or EV_LOCK.exists()):
    EVAL = {"scored": False, "reason": f"form {BEST} was not kept (development {SC[BEST][0]:.4f} against {THR[0]:.4f}, confirmation "
                                         f"{SC[BEST][1]:.4f} against {THR[1]:.4f}); by the rules the evaluation set is not scored"}
    print("the evaluation set is NOT scored:", EVAL["reason"])
else:
    EV_DIR.mkdir(exist_ok=True)
    if EV_RES.exists():
        EVAL = json.loads(EV_RES.read_text())
        print(f"the evaluation set was scored on {EVAL['scores_written']}; its saved results are read, and nothing is scored again")
    else:
        if not EV_LOCK.exists():
            files = []
            for t in TYPES:
                for q in machines(t, "eval"):
                    te = dc_rows(t, "eval", "test", q); sc = score_of(BEST, t, "eval", q, SEEDS)
                    assert np.isfinite(sc).all(), (t, q)
                    f = EV_DIR / f"anomaly_score_{t}_{q}.csv"
                    f.write_text("".join(f"{a},{float(b)!r}\n" for a, b in sorted(zip([DMETA[t]["name"][i] for i in te], sc))))
                    files.append(f.name)
                FM.clear(); VM.clear()
            EV_LOCK.write_text(json.dumps({"written": time.strftime("%Y-%m-%d %H:%M"), "files": files, "test_run": TEST,
                                           "system": {"form": BEST, "view_seeds": SEEDS, "frozen": [[e, sd] for e, _, sd in FROZEN],
                                                      "calibration": CAL if BEST in ("Z", "ZP") else None}}))
            print(f"the evaluation scores of the kept system (form {BEST}) are written ({len(files)} files), before any label is read", flush=True)
        lock = json.loads(EV_LOCK.read_text())
        with open(get_file(DC_GT), newline="") as fh:                 # the labels: downloaded and read only here
            gt_rows = list(csv.reader(fh))
        GT, cur = {}, None
        for r in gt_rows:
            if not r:
                continue
            if len(r) < 2:                                            # a machine-type line, as in the official evaluator
                cur = r[0].strip(); continue
            GT[(cur, r[0].strip())] = int(float(r[2]))
        per = {}
        for fn in lock["files"]:
            t, q = re.match(r"anomaly_score_(\w+?)_(id_\d+)\.csv$", fn).groups()
            rows = [ln.split(",") for ln in (EV_DIR / fn).read_text().splitlines() if ln]
            missing = [a for a, _ in rows if (t, a) not in GT]
            assert not missing, f"no label for {missing[:3]}"
            per.setdefault(t, {})[q] = dc_metrics(np.array([GT[(t, a)] for a, _ in rows]), np.array([float(b) for _, b in rows]))
        EVAL = {"scored": True, "scores_written": lock["written"], "results_written": time.strftime("%Y-%m-%d %H:%M"), "system": lock["system"],
                "test_run": TEST, "ground_truth": list(DC_GT[1:]),
                "per_type": {t: {"per_machine": per[t], "AUC": float(np.mean([v["AUC"] for v in per[t].values()])),
                                 "pAUC": float(np.mean([v["pAUC"] for v in per[t].values()]))} for t in TYPES}}
        EVAL["mean"] = {"AUC": round(float(np.mean([EVAL["per_type"][t]["AUC"] for t in TYPES])), 4),
                        "pAUC": round(float(np.mean([EVAL["per_type"][t]["pAUC"] for t in TYPES])), 4)}
        EV_RES.write_text(json.dumps(EVAL, indent=1))
    if EVAL["system"]["form"] != BEST or not KEPT:
        print(f"NOTE: the evaluation set was scored in an earlier session with form {EVAL['system']['form']}; this session selects {BEST} "
              f"(kept: {KEPT}). The saved results are reported, and nothing is scored again.")
    print("\nEvaluation set (a second scoring; notebook 23's stands as the program's T3) · AUC / pAUC:")
    for t in TYPES:
        e = EVAL["per_type"][t]
        print(f"  {NAME[t]:<11} kept system {e['AUC']:.4f} / {e['pAUC']:.4f} · notebook 23 {EV23[t][0]:.4f} / {EV23[t][1]:.4f} · "
              f"change {e['AUC'] - EV23[t][0]:+.4f} · best of the five teams {BEST5[t] / 100:.4f} · "
              f"mean of AUC and pAUC {50 * (e['AUC'] + e['pAUC']):.2f} (AnoPatch {ANOPATCH['evaluation'][t]:.2f})", flush=True)
    print(f"  mean AUC over the four types {EVAL['mean']['AUC']} (notebook 23: {round(float(np.mean([EV23[t][0] for t in TYPES])), 4)}) · "
          f"mean pAUC {EVAL['mean']['pAUC']} (notebook 23: {round(float(np.mean([EV23[t][1] for t in TYPES])), 4)})")
    try:
        OFFICIAL = official_check()
    except Exception as e:                                            # the result above does not depend on this check
        OFFICIAL = {"ran": False, "reason": f"an unexpected error ({repr(e)[:200]})"}
    (EV_DIR / "official_evaluator_check.json").write_text(json.dumps(OFFICIAL, indent=1))
    if OFFICIAL["ran"]:
        print(f"the official evaluator gives the same numbers: {OFFICIAL['agree']} (largest difference {OFFICIAL['max_abs_difference']:.1e} "
              f"over {OFFICIAL['values_compared']} values)")
        if not OFFICIAL["agree"]:
            print("  the numbers differ: by the rules, the official evaluator's are the result:", OFFICIAL["per_type"])
    else:
        print("the official evaluator's check was NOT run:", OFFICIAL["reason"])

In [ ]:
# 10 · Save, one picture, and one zip to download
import matplotlib.pyplot as plt

RESULT = {"rules": RULES, "date": time.strftime("%Y-%m-%d %H:%M"), "test_run": TEST,
          "meta": {"device": DEVICE, "gpu": GPU, "gpu_gb": round(GPU_GB, 1), "torch": torch.__version__, "torchaudio": torchaudio.__version__,
                   "reference": {"repository": "https://github.com/soonhyeon/Noisy-ArcMix", "commit": REF_COMMIT, "licence": "MIT, Copyright (c) 2023 soonhyeon"},
                   "sv": SV, "n_frames": int(N_FRAMES), "last_kernel": list(LAST_KERNEL), "classes": CLASSES, "n_clips": int(len(ITEMS)),
                   "zips": {"development": {k: list(v[1:]) for k, v in DC_DEV.items()}, "additional": {k: list(v[1:]) for k, v in XZIPS.items()},
                            "evaluation": {k: list(v[1:]) for k, v in DC_EVAL.items()}, "ground_truth": list(DC_GT[1:])},
                   "memory_layout_seconds_per_step": MF_TIMES, "prep_seconds": PREP_S, "train_seconds": T_TRAIN, "embed_seconds": T_EMBED,
                   "session_minutes": round((time.time() - T_START) / 60, 1), "drive_free_gb": round(DRIVE_FREE_GB, 1)},
          "G0": G0, "G1": G1, "G_VIEW": {"first_seed": GV0, "three_seeds": GV},
          "counts": DCASE["counts"],
          "development": {"systems": DEV, "scores": {nm: list(SC[nm]) for nm in SYSTEMS}, "per_seed": PER_SEED, "preview_first_seed": PREVIEW},
          "calibration": CAL, "decision": DECISION, "evaluation": EVAL, "official_evaluator": OFFICIAL,
          "notebook_23": {"development": DEV23, "evaluation": EV23, "current": list(CURRENT), "keep_at": list(KEEP_AT)},
          "yardsticks": {"published_development": PUBLISHED_DEV, "published_worst_machine": PUBLISHED_MAUC, "anopatch": ANOPATCH,
                         "best_of_five_teams_evaluation": BEST5, "teams": TEAMS, "dcase_baseline_development": DCASE_BASELINE_DEV},
          "training_logs": LOGS}
(OUT / "second_view.json").write_text(json.dumps(RESULT, indent=1, default=float, ensure_ascii=False))
# every development test clip's score under every system, so that each development number above can be recomputed from the zip
SC_DIR = OUT / "development_scores"; SC_DIR.mkdir(exist_ok=True)
for (t, q), d in DEV_SC.items():
    (SC_DIR / f"scores_{t}_{q}.csv").write_text("file,label," + ",".join(SYSTEMS) + "\n" + "".join(
        f"{d['file'][i]},{int(d['label'][i])}," + ",".join(repr(float(d[nm][i])) for nm in SYSTEMS) + "\n" for i in range(len(d["file"]))))

INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
COL = {"frozen": "#2a78d6", "view": "#eb6834", "fused": "#1baf7a"}    # three categorical slots, checked with the palette validator


MARK = {"frozen": "o", "view": "s", "fused": "D"}                    # a shape per system as well as a colour


def dots(ax, xs, vals, k, lab):
    ax.plot(xs, vals, MARK[k], ms=9, color=COL[k], mec="white", mew=1.2, label=lab, zorder=3)
    for a, v in zip(xs, vals):                                        # every mark carries its value
        ax.text(a, v + 0.006 * (ax.get_ylim()[1] - ax.get_ylim()[0]) / 0.3, f"{v:.3f}", ha="center", va="bottom", fontsize=7.5, color=INK)


fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.6))
x, w = np.arange(len(TYPES)), 0.27
ax = axes[0]
series = [("frozen", "frozen system (notebook 23)", [DEV["frozen"][t]["AUC"] for t in TYPES]),
          ("view", "second view alone (LDN1)", [DEV["view_LDN1"][t]["AUC"] for t in TYPES]),
          ("fused", f"fused, form {BEST}" + (" (kept)" if KEPT else " (not kept)"), [DEV[BEST][t]["AUC"] for t in TYPES])]
pub = [PUBLISHED_DEV["Noisy-ArcMix, TASTgram-MFN (arXiv 2310.06364)"][t][0] / 100 for t in TYPES]
lo = math.floor((min([v for _, _, vals in series for v in vals] + pub) - 0.02) * 50) / 50
ax.set_ylim(lo, 1.0 + 0.06 * (1.0 - lo))
for j, (k, lab, vals) in enumerate(series):
    dots(ax, x + (j - 1) * w, vals, k, lab)
ax.hlines(pub, x - 0.43, x + 0.43, color=INK, lw=1.4, label="Noisy-ArcMix as published (41 classes; not like for like)", zorder=2)
ax.set_ylabel("AUC (mean over a type's machines); the scale does not start at zero", color=MUTED)
ax.set_title(f"Development set · form {BEST}: development {SC[BEST][0]:.4f} (needs {THR[0]:.4f}), confirmation {SC[BEST][1]:.4f} (needs {THR[1]:.4f})",
             loc="left", color=INK, fontsize=9.5)
ax = axes[1]
ev_old, best5 = [EV23[t][0] for t in TYPES], [BEST5[t] / 100 for t in TYPES]
ev_new = [EVAL["per_type"][t]["AUC"] for t in TYPES] if EVAL["scored"] else []
lo2 = math.floor((min(ev_old + ev_new + best5) - 0.02) * 50) / 50
ax.set_ylim(lo2, 1.0 + 0.06 * (1.0 - lo2))
if EVAL["scored"]:
    dots(ax, x - w / 2, ev_old, "frozen", "frozen system (notebook 23, the program's T3)")
    dots(ax, x + w / 2, ev_new, "fused", f"kept system, form {EVAL['system']['form']} (a second scoring)")
    ttl = f"Evaluation set · mean AUC {EVAL['mean']['AUC']} (notebook 23: {round(float(np.mean(ev_old)), 4)})"
else:
    dots(ax, x, ev_old, "frozen", "frozen system (notebook 23, the program's T3)")
    ttl = "Evaluation set · not scored again: the view was not kept"
ax.hlines(best5, x - 0.43, x + 0.43, color=INK, lw=1.4, label="best of the program's five 2020 teams, per type", zorder=2)
ax.set_title(ttl, loc="left", color=INK, fontsize=9.5)
for ax in axes:
    ax.set_xticks(x, [NAME[t] for t in TYPES]); ax.set_xlim(-0.6, len(TYPES) - 0.4)
    ax.grid(axis="y", color=GRID, lw=0.6); ax.set_axisbelow(True); ax.tick_params(colors=MUTED)
    ax.legend(frameon=False, fontsize=8.5, loc="upper center", bbox_to_anchor=(0.5, -0.09), ncol=2)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
fig.suptitle("Notebook 25 · a second view (TASTgram-MFN with Noisy-ArcMix, from scratch) for the single-microphone system on DCASE 2020", fontsize=11)
fig.tight_layout(rect=(0, 0, 1, 0.95)); fig.savefig(OUT / "second_view_dcase2020.png", dpi=150); plt.show()

ZIP = OUT / "p04_n25_results.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for f in ("second_view.json", "second_view_dcase2020.png"):
        z.write(OUT / f, f)
    for f in sorted(SC_DIR.glob("scores_*.csv")):
        z.write(f, f"{SC_DIR.name}/{f.name}")
    if EVAL["scored"]:
        for f in sorted(EV_DIR.glob("*")):
            z.write(f, f"{EV_DIR.name}/{f.name}")
print(f"saved {ZIP} · {(time.time() - T_START) / 60:.0f} min in this session")
if IN_COLAB:
    from google.colab import files
    files.download(str(ZIP))

## Send back
Send the zip that downloaded (`p04_n25_results.zip`). It is also in Drive › `P04_SecondView`.